In [1]:
# 1. Instalação das bibliotecas necessárias
!pip install plotly numpy

# 2. Importação dos módulos
import numpy as np
import plotly.graph_objects as go

# Parâmetros de rede (em Angstroms)
a, b, c = 5.423, 7.675, 5.461

# Distâncias de ligação do octaedro
d_apical = 1.94262
d_eq_short = 1.89720
d_eq_long = 2.00396

def get_octahedron_vertices(center):
    cx, cy, cz = center
    return np.array([
        [cx, cy, cz + d_apical],
        [cx, cy, cz - d_apical],
        [cx + d_eq_short, cy, cz],
        [cx - d_eq_short, cy, cz],
        [cx, cy + d_eq_long, cz],
        [cx, cy - d_eq_long, cz]
    ])

# Posições dos sítios
B_positions = [
    [0, 0, 0], [a, 0, 0], [0, b, 0], [a, b, 0],
    [0, 0, c], [a, 0, c], [0, b, c], [a, b, c],
    [a/2, b/2, 0], [a/2, b/2, c], [a/2, 0, c/2], [0, b/2, c/2], [a, b/2, c/2]
]

A_positions = [
    [a*0.5, b*0.25, c*0.5], [a*0.5, b*0.75, c*0.5],
    [0, b*0.25, c*0.5], [a, b*0.75, c*0.5]
]

fig = go.Figure()

# Índices para montagem das faces dos octaedros
faces_i = [0, 0, 0, 0, 1, 1, 1, 1]
faces_j = [2, 3, 4, 5, 2, 3, 4, 5]
faces_k = [4, 4, 2, 3, 5, 5, 2, 3]

# Renderização dos octaedros BO6 e átomos de Oxigênio
for idx, B_pos in enumerate(B_positions):
    verts = get_octahedron_vertices(B_pos)

    fig.add_trace(go.Mesh3d(
        x=verts[:, 0], y=verts[:, 1], z=verts[:, 2],
        i=faces_i, j=faces_j, k=faces_k,
        color='#2ecc71', opacity=0.35,
        name='Octaedro BO6', showlegend=(idx == 0)
    ))

    fig.add_trace(go.Scatter3d(
        x=verts[:, 0], y=verts[:, 1], z=verts[:, 2],
        mode='markers', marker=dict(size=5, color='#e74c3c'),
        name='Oxigenio (O)', showlegend=(idx == 0)
    ))

# Átomos dos Sítios A e B
B_arr, A_arr = np.array(B_positions), np.array(A_positions)

fig.add_trace(go.Scatter3d(
    x=B_arr[:, 0], y=B_arr[:, 1], z=B_arr[:, 2],
    mode='markers', marker=dict(size=8, color='#1e8449'), name='Sitio B (Co/Fe/Mn)'
))

fig.add_trace(go.Scatter3d(
    x=A_arr[:, 0], y=A_arr[:, 1], z=A_arr[:, 2],
    mode='markers', marker=dict(size=14, color='#2980b9'), name='Sitio A (La/Ca)'
))

# Animação de rotação em 360 graus
angles = np.linspace(0, 2*np.pi, 60)
fig.frames = [
    go.Frame(
        layout=dict(scene_camera=dict(
            eye=dict(x=2.2*np.cos(a_rad), y=2.2*np.sin(a_rad), z=1.2),
            center=dict(x=a/2, y=b/2, z=c/2)
        )),
        name=f'f_{i}'
    ) for i, a_rad in enumerate(angles)
]

# Configurações de layout e controles interativos
fig.update_layout(
    title='Estrutura 3D Interativa La1.5Ca0.5Co(Fe0.5Mn0.5)O6',
    scene=dict(xaxis_title='a (A)', yaxis_title='b (A)', zaxis_title='c (A)', aspectmode='data'),
    updatemenus=[dict(
        type="buttons", showactive=False, x=0.05, y=0.95,
        buttons=[
            dict(label="Iniciar Rotacao", method="animate",
                 args=[None, {"frame": {"duration": 50, "redraw": True}, "fromcurrent": True, "loop": True}]),
            dict(label="Pausar", method="animate",
                 args=[[None], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate"}])
        ]
    )]
)

# Exportação do arquivo interativo e exibição
fig.write_html("perovskita_3d_interativa.html")
fig.show()